In [1]:
import pandas as pd
import numpy as np
import torch
import pickle
import os
import json
import gc
from torch.distributions import Bernoulli
from torch.optim import LBFGS
from tqdm import tqdm
from scipy.stats import pearsonr
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import Manager
import multiprocessing as mp

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tueplots import bundles
bundles.icml2024()

from torchmetrics import AUROC
auroc = AUROC(task="binary")

import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(0)

device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

def visualize_response_matrix(results, value, filename):
    # Extract the groups labels in the order of the columns
    group_values = results.columns.get_level_values("scenario")

    # Identify the boundaries where the group changes
    boundaries = []
    for i in range(1, len(group_values)):
        if group_values[i] != group_values[i - 1]:
            boundaries.append(i - 0.5)  # using 0.5 to place the line between columns

    # Visualize the results with a matrix: red is 0, white is -1 and blue is 1
    cmap = mcolors.ListedColormap(["white", "red", "blue"])
    bounds = [-1.5, -0.5, 0.5, 1.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    # Calculate midpoints for each group label
    groups_list = list(group_values)
    group_names = []
    group_midpoints = []
    current_group = groups_list[0]
    start_index = 0
    for i, grp in enumerate(groups_list):
        if grp != current_group:
            midpoint = (start_index + i - 1) / 2.0
            group_names.append(current_group)
            group_midpoints.append(midpoint)
            current_group = grp
            start_index = i
    # Add the last group
    midpoint = (start_index + len(groups_list) - 1) / 2.0
    group_names.append(current_group)
    group_midpoints.append(midpoint)

    # Define the minimum spacing between labels (e.g., 100 units)
    min_spacing = 100
    last_label_pos = -float("inf")
    # Plot the matrix
    with plt.rc_context(bundles.icml2024(usetex=True, family="serif")):
        fig, ax = plt.subplots(figsize=(20, 10))
        cax = ax.matshow(value, aspect="auto", cmap=cmap, norm=norm)

        # Add vertical lines at each boundary
        for b in boundaries:
            ax.axvline(x=b, color="black", linewidth=0.25, linestyle="--", alpha=0.5)
        
        # Add group labels above the matrix, only if they're spaced enough apart
        for name, pos in zip(group_names, group_midpoints):
            if pos - last_label_pos >= min_spacing:
                ax.text(pos, -5, name, ha='center', va='bottom', rotation=90, fontsize=3)
                last_label_pos = pos

        # Add model labels on the y-axis
        ax.set_yticks(range(len(results.index)))
        ax.set_yticklabels(results.index, fontsize=3)

        # Add a colorbar
        cbar = plt.colorbar(cax)
        cbar.set_ticks([-1, 0, 1])
        cbar.set_ticklabels(["-1", "0", "1"])
        plt.savefig(filename, dpi=600, bbox_inches="tight")
        plt.close()

def trainer(parameters, optim, closure, n_iter=100, verbose=True):
    pbar = tqdm(range(n_iter)) if verbose else range(n_iter)
    for iteration in pbar:
        if iteration > 0:
            previous_parameters = [p.clone() for p in parameters]
            previous_loss = loss.clone()
        
        loss = optim.step(closure)
        
        if iteration > 0:
            d_loss = (previous_loss - loss).item()
            d_parameters = sum(
                torch.norm(prev - curr, p=2).item()
                for prev, curr in zip(previous_parameters, parameters)
            )
            grad_norm = sum(torch.norm(p.grad, p=2).item() for p in parameters if p.grad is not None)
            if verbose:
                pbar.set_postfix({"grad_norm": grad_norm, "d_parameter": d_parameters, "d_loss": d_loss})
            
            if d_loss < 1e-5 and d_parameters < 1e-5 and grad_norm < 1e-5:
                break
    return parameters

def compute_auc(probs, data, train_idtor, test_idtor):
    train_probs = probs[train_idtor.bool()]
    test_probs = probs[test_idtor.bool()]
    train_labels = data[train_idtor.bool()]
    test_labels = data[test_idtor.bool()]
    train_auc = auroc(train_probs, train_labels)
    test_auc = auroc(test_probs, test_labels)
    print(f"train auc: {train_auc}")
    print(f"test auc: {test_auc}")
    
    return train_auc, test_auc

def compute_cttcorr(probs, data, train_idtor, test_idtor):
    train_probs  = probs.clone()
    test_probs   = probs.clone()
    train_labels = data.clone()
    test_labels  = data.clone()

    train_mask = ~train_idtor.bool()
    train_probs[train_mask]  = float('nan')
    train_labels[train_mask] = float('nan')

    test_mask = ~test_idtor.bool()
    test_probs[test_mask]   = float('nan')
    test_labels[test_mask]  = float('nan')
    
    train_prob_ctt = torch.nanmean(train_probs, dim=1).detach().cpu().numpy()
    train_label_ctt = torch.nanmean(train_labels, dim=1).detach().cpu().numpy()
    train_mask = ~np.isnan(train_prob_ctt) & ~np.isnan(train_label_ctt)
    train_cttcorr = pearsonr(train_prob_ctt[train_mask], train_label_ctt[train_mask]).statistic
    
    test_prob_ctt = torch.nanmean(test_probs, dim=1).detach().cpu().numpy()
    test_label_ctt = torch.nanmean(test_labels, dim=1).detach().cpu().numpy()
    test_mask = ~np.isnan(test_prob_ctt) & ~np.isnan(test_label_ctt)
    test_cttcorr = pearsonr(test_prob_ctt[test_mask], test_label_ctt[test_mask]).statistic
    
    print(f"train cttcorr: {train_cttcorr}")
    print(f"test cttcorr: {test_cttcorr}")

    return train_cttcorr, test_cttcorr

In [2]:
with open(f"../data/resmat.pkl", "rb") as f:
    results = pickle.load(f)

dtype = torch.float64 if device.startswith("cuda") else torch.float32

# data_withnan, missing=nan
# data_withneg1, missing=-1
# data_with0, missing=0
data_withnan = torch.tensor(results.values, dtype=dtype, device=device)
data_idtor = (~torch.isnan(data_withnan)).to(dtype)
data_withneg1 = data_withnan.nan_to_num(nan=-1.0)
data_with0 = data_withneg1 * data_idtor
data_with0 = data_with0.nan_to_num(nan=0.0)
n_test_takers, n_items = data_with0.shape
scenarios = results.columns.get_level_values("scenario").unique()

In [3]:
master_test_df = pd.read_csv("../data/master_test_metadata.csv")
final_test_data_matrix = pd.read_pickle("../data/master_test_data_matrix_factor.pkl").values
dtype = torch.float64 if device.startswith("cuda") else torch.float32
final_test_data_matrix = np.nan_to_num(final_test_data_matrix, nan=0)

In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from scipy.special import expit # Numerically stable sigmoid function
from scipy.stats import pearsonr

# --- PyTorch and Metrics Imports (from your reference code) ---
import torch
from torchmetrics import AUROC
auroc = AUROC(task="binary")
# --- End of Imports ---
# ===================================================================
# == Step 1: Load Data and Create Train/Test Split
# ===================================================================
print("Loading response matrix...")
resmat = pd.read_pickle("../data/resmat.pkl")

Loading response matrix...


In [5]:
import pickle
import numpy as np
import pandas as pd
from scipy.special import expit
import torch
import sys
sys.path.append('../mirt-official')
from load_params import load_and_rotate

# Create train/test split
non_nan_indices = np.argwhere(resmat.notna().values)
np.random.seed(42)
np.random.shuffle(non_nan_indices)

test_size = int(len(non_nan_indices) * 0.20)
test_indices = non_nan_indices[:test_size]
train_indices = non_nan_indices[test_size:]

print(f"Split data into {len(train_indices)} train samples and {len(test_indices)} test samples.")

# ===================================================================
# == Step 2: Load MIRT Model Parameters
# ===================================================================
print(f"\nLoading MIRT model parameters...")

# Load and rotate MIRT parameters using load_params.py
theta_z_scores, a_rotated, b_adjusted = load_and_rotate('../mirt-official/output/mirt_model_k6.pt')

print(f"Loaded MIRT model with {theta_z_scores.shape[1]} factors")
print(f"Subject abilities (theta) shape: {theta_z_scores.shape}")
print(f"Item parameters (a) shape: {a_rotated.shape}")
print(f"Item difficulties (b) shape: {b_adjusted.shape}")

# Use MIRT parameters instead of SVD
subject_scores = theta_z_scores  # Use rotated and standardized theta
all_item_factors_T = a_rotated.T  # Transpose for compatibility with downstream code

# Reconstruct the full matrix using MIRT 2PL model
# logits = theta @ a.T - b
reconstructed_matrix = subject_scores @ all_item_factors_T - b_adjusted[None, :]
probs_matrix_np = expit(reconstructed_matrix)

print(f"Successfully reconstructed the full matrix of shape: {reconstructed_matrix.shape}")

# ===================================================================
# == Step 3: Prepare Data for Evaluation
# ===================================================================
data_np = resmat.fillna(0).values

# Create boolean masks (numpy)
train_idtor_np = np.zeros_like(data_np, dtype=bool)
test_idtor_np = np.zeros_like(data_np, dtype=bool)
train_idtor_np[train_indices[:, 0], train_indices[:, 1]] = True
test_idtor_np[test_indices[:, 0], test_indices[:, 1]] = True

# Convert NumPy arrays to PyTorch tensors for evaluation
print("\nConverting results to PyTorch Tensors for evaluation...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"

probs_tensor = torch.tensor(probs_matrix_np, dtype=torch.float32, device=device)
data_tensor = torch.tensor(data_np, dtype=torch.float32, device=device)
train_idtor_tensor = torch.tensor(train_idtor_np, dtype=torch.int, device=device)
test_idtor_tensor = torch.tensor(test_idtor_np, dtype=torch.int, device=device)

print("\n--- Running MIRT Model Evaluation ---")
compute_auc(probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)
compute_cttcorr(probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)

Split data into 4267820 train samples and 1066954 test samples.

Loading MIRT model parameters...
--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 6])
Original 'a' matrix shape: torch.Size([78712, 6])

--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 6)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 6)

--- Final Standardized Z-Scores (from transformed theta) ---
These are the scores you should use for interpretation.
[[ 1.5905465   3.49991235 -2.70676842  0.80146618  0.72562385 -0.62409166]
 [ 1.26710786 -0.50314049  1.10198573 -1.1741239   0.34140497  0.07110775]
 [ 1.86002922  0.60271614  0.90773613  0.07469808 -0.76376825 -1.55688678]
 ...
 [-1.05323766 -0.53844968 -2.76657604  1.42263159  0.66665258  2.97106778]
 [-0.43951293 -0.54343183 -2.60149674  2.23806708  0.32196824  3.12436554]
 [ 0.04734164 -0.47165056 -1.30415358  1.18832103  1.00318341  2.02381296]]
Loaded MIRT model with 6 factors
Subject abilities (theta) shape

(np.float32(0.99409676), np.float32(0.9837507))

In [6]:
# The subject_scores variable is available from the MIRT model.
# It has the correct shape (n_users, n_factors), e.g., (183, 6).
# Use MIRT parameters instead of SVD
mirt_model_abilities = subject_scores

# The MIRT item discrimination parameters are already available as a_rotated
# Shape: (n_items, n_factors)
mirt_item_params = a_rotated

In [7]:
print(f"MIRT parameters shapes:")
print(f"Subject scores (theta): {subject_scores.shape}")
print(f"Item discriminations (a): {a_rotated.shape}")  
print(f"Item difficulties (b): {b_adjusted.shape}")

MIRT parameters shapes:
Subject scores (theta): (183, 6)
Item discriminations (a): (78712, 6)
Item difficulties (b): (78712,)


In [8]:
print("\n--- Step 3: Training Neural Network to predict item parameters ---")

with open("../data/embed_meta-llama_Llama-3.1-8B-Instruct.pkl", "rb") as f:
    df_embed = pickle.load(f)


--- Step 3: Training Neural Network to predict item parameters ---


In [9]:
# ===================================================================
# == Step 3: Prepare NN Training Data with MIRT Parameters (a + b)
# ===================================================================

question_to_emb = dict(zip(df_embed["question"], df_embed["embedding"]))
questions = resmat.columns.get_level_values("input.text").tolist()
embeds = [question_to_emb.get(q, None) for q in questions]

# Combine discrimination parameters (a) and difficulty parameters (b)
# a_rotated shape: (n_items, n_factors), b_adjusted shape: (n_items,)
# Combined target shape: (n_items, n_factors + 1)
y_combined_all = np.concatenate([a_rotated, b_adjusted.reshape(-1, 1)], axis=1)

combined_data = pd.DataFrame({'embedding': embeds, 'params_vector': list(y_combined_all)})
cleaned_data = combined_data.dropna(subset=['embedding'])
X_embeddings = np.vstack(cleaned_data['embedding'].values)
y_combined_params = np.vstack(cleaned_data['params_vector'].values)

n_factors = a_rotated.shape[1]
print(f"Prepared NN training data: {X_embeddings.shape[0]} items with embeddings")
print(f"Target parameters shape: {y_combined_params.shape} (items x [a_params + b_param])")
print(f"Number of factors: {n_factors}")
print(f"Predicting {n_factors} discrimination params + 1 difficulty param per item")


Prepared NN training data: 78688 items with embeddings
Target parameters shape: (78688, 7) (items x [a_params + b_param])
Number of factors: 6
Predicting 6 discrimination params + 1 difficulty param per item


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm

# data_idtor = train_idtor + test_idtor
# apply random train/test mask to the matrix, and ensure no one row or column is fully masked
valid_condition = False
trial = 0
while not valid_condition:
    train_idtor = torch.bernoulli(data_idtor * 0.8).int()
    test_idtor = data_idtor - train_idtor
    valid_condition = (train_idtor.sum(axis=1) != 0).all() and (train_idtor.sum(axis=0) != 0).all()
    print(f"trial {trial} valid condition: {valid_condition}")
    trial += 1

class ItemParameterPredictor(nn.Module):
    def __init__(self, e, n): 
        super(ItemParameterPredictor, self).__init__();
        self.network = nn.Sequential(nn.Linear(e, 1024),
                                     nn.ReLU(), nn.Dropout(0.5),
                                     nn.Linear(1024, 512),
                                     nn.ReLU(), nn.Dropout(0.5),
                                     nn.Linear(512, 128),
                                     nn.ReLU(),
                                     nn.Dropout(0.5),
                                     nn.Linear(128, n))
    def forward(self, x): return self.network(x)

# Note: We train the NN on ALL available items, because the target (y_combined_params)
# was already derived purely from the training set, so there is no data leakage.
X_train_t = torch.tensor(X_embeddings, dtype=torch.float32)
y_train_t = torch.tensor(y_combined_params, dtype=torch.float32)
train_dataset = TensorDataset(X_train_t, y_train_t); train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
nn_model = ItemParameterPredictor(X_embeddings.shape[1], y_combined_params.shape[1]); nn_model.to(device)
loss_fn = nn.MSELoss(); optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.0001)

n_epochs = 200
for epoch in range(n_epochs):
    nn_model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        p = nn_model(X_batch); loss = loss_fn(p, y_batch)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{n_epochs} | Average Training Loss: {avg_loss:.6f}")
print("--- NN Training Complete ---")

# ===================================================================
# == Step 4: Reconstruct and Evaluate with the Unified Split
# ===================================================================
print("\n--- Step 4: Reconstructing matrix and running final evaluation ---")

embed_dim = X_embeddings.shape[1]
full_embeds_np = np.zeros((len(embeds), embed_dim))
for i, emb in enumerate(embeds):
    if emb is not None: full_embeds_np[i] = emb

nn_model.eval()
with torch.no_grad():
    input_tensor = torch.tensor(full_embeds_np, dtype=torch.float32).to(device)
    nn_predicted_item_params = nn_model(input_tensor).cpu().numpy()

# Split predicted parameters back into a (discrimination) and b (difficulty) components
# nn_predicted_item_params shape: (n_items, n_factors + 1)
nn_predicted_a = nn_predicted_item_params[:, :n_factors]  # First n_factors columns are discrimination
nn_predicted_b = nn_predicted_item_params[:, -1]         # Last column is difficulty

# Reconstruct the logit matrix using MIRT 2PL model with BOTH predicted a and b
# logits = theta @ a.T - b
reconstructed_matrix_from_nn = subject_scores @ nn_predicted_a.T - nn_predicted_b[None, :]
# Convert logits to probabilities  
probs_matrix_from_nn = expit(reconstructed_matrix_from_nn)

print(f"Using predicted discrimination parameters shape: {nn_predicted_a.shape}")
print(f"Using predicted difficulty parameters shape: {nn_predicted_b.shape}")

# Prepare all final tensors for evaluation
probs_tensor = torch.tensor(probs_matrix_from_nn, dtype=torch.float32, device=device)
data_tensor = torch.tensor(resmat.fillna(0).values, dtype=torch.float32, device=device)
train_idtor_tensor = train_idtor.to(device)
test_idtor_tensor = test_idtor.to(device)

# Run the final evaluations using the unified train/test split
print("\n--- Final Evaluation of MIRT + NN-based Reconstruction (predicting a + b) ---")
compute_auc(probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)
compute_cttcorr(probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)

trial 0 valid condition: True
Epoch 1/200 | Average Training Loss: 0.545659
Epoch 2/200 | Average Training Loss: 0.476089
Epoch 3/200 | Average Training Loss: 0.457236
Epoch 4/200 | Average Training Loss: 0.446309
Epoch 5/200 | Average Training Loss: 0.436027
Epoch 6/200 | Average Training Loss: 0.428387
Epoch 7/200 | Average Training Loss: 0.421742
Epoch 8/200 | Average Training Loss: 0.415366
Epoch 9/200 | Average Training Loss: 0.410048
Epoch 10/200 | Average Training Loss: 0.403723
Epoch 11/200 | Average Training Loss: 0.400110
Epoch 12/200 | Average Training Loss: 0.394144
Epoch 13/200 | Average Training Loss: 0.389910
Epoch 14/200 | Average Training Loss: 0.385501
Epoch 15/200 | Average Training Loss: 0.380580
Epoch 16/200 | Average Training Loss: 0.376683
Epoch 17/200 | Average Training Loss: 0.372977
Epoch 18/200 | Average Training Loss: 0.369950
Epoch 19/200 | Average Training Loss: 0.365990
Epoch 20/200 | Average Training Loss: 0.361679
Epoch 21/200 | Average Training Loss: 0

(np.float32(0.9909957), np.float32(0.98799133))

In [11]:
torch.save(train_idtor, os.path.join('../result/', "train_idtor.pt"))
torch.save(test_idtor, os.path.join('../result/', "test_idtor.pt"))
np.save(os.path.join('../result/', "nn_predicted_a_params.npy"), nn_predicted_a)
np.save(os.path.join('../result/', "nn_predicted_b_params.npy"), nn_predicted_b)

In [12]:
# After the training loop is finished
torch.save(nn_model.state_dict(), '../result/mirt_item_parameter_predictor_model.pth')
print("\n--- Final MIRT-based model saved to mirt_item_parameter_predictor_model.pth ---")


--- Final MIRT-based model saved to mirt_item_parameter_predictor_model.pth ---


# Visualise

In [14]:
import sys, os
import numpy as np
import torch
sys.path.append('../mirt-official')
from load_params import load_and_rotate
theta, mirt_a, mirt_b = load_and_rotate('../mirt-official/output/mirt_model_k6.pt')
nn_predicted_a = np.load(os.path.join('../result/', "nn_predicted_a_params.npy"))
nn_predicted_b = np.load(os.path.join('../result/', "nn_predicted_b_params.npy"))
train_idtor = torch.load(os.path.join('../result/', "train_idtor.pt"), map_location=torch.device('cpu'))
test_idtor = torch.load(os.path.join('../result/', "test_idtor.pt"), map_location=torch.device('cpu'))

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 6])
Original 'a' matrix shape: torch.Size([78712, 6])

--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 6)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 6)

--- Final Standardized Z-Scores (from transformed theta) ---
These are the scores you should use for interpretation.
[[ 1.5905465   3.49991235 -2.70676842  0.80146618  0.72562385 -0.62409166]
 [ 1.26710786 -0.50314049  1.10198573 -1.1741239   0.34140497  0.07110775]
 [ 1.86002922  0.60271614  0.90773613  0.07469808 -0.76376825 -1.55688678]
 ...
 [-1.05323766 -0.53844968 -2.76657604  1.42263159  0.66665258  2.97106778]
 [-0.43951293 -0.54343183 -2.60149674  2.23806708  0.32196824  3.12436554]
 [ 0.04734164 -0.47165056 -1.30415358  1.18832103  1.00318341  2.02381296]]
